# 05 — Fresh portfolio backtest from current forecasts

Run notebooks **01 → 02 → 03 → 04** to completion first, then execute every cell here in order using the project virtual environment kernel (**Python 3 / Python (DSA5205)**). This notebook computes the original dollar-neutral sector–SPY pair strategy from the current seven-column prediction exports. It does not load historical backtest caches or require the external `5205proj2.md`.

Each execution creates a unique directory in `results/reproduction/`. Existing results are preserved. Trading, sizing and accounting engines remain unchanged. Validation selects the common threshold; test uses that threshold without further selection. These experiments retain the project's prior exploratory provenance.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "requirements.txt").is_file()
             and (p / "notebooks").is_dir() and (p / "scripts").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project directory or a subdirectory.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import fresh_backtest_pipeline as pipeline
print("Project:", ROOT)
print("Environment:", pipeline.runtime())


## 1. Register this run and its rules

Start a new run before producing any returns. Costs: 3 bp per net traded side, annual borrow 1%, no interest on short collateral. At most four pairs, each with a 12.5% sector leg and opposite netted SPY leg. Validation entry candidates are fixed at 5/10/15/20/30 bp; exit is half entry.

`RUN_COST_SENSITIVITIES=True` recomputes all eight cost cases. Setting it to False is an explicitly shorter demonstration, not a complete sensitivity reproduction.


In [ ]:
RUN_COST_SENSITIVITIES = True
ctx = pipeline.start_pair_run(ROOT)
RUN_DIR = ctx["out"]
display(ctx["config"]["protocol"])


## 2. Read and validate CURRENT upstream files; generate NEW signals

Check all six model/scheme forecast key sets against the current common input mask. Reconstruct the next-open calendar, verify panel timing, then generate as-of trading instructions. Future labels and scoring flags are excluded from the trading signal interface. Missing signals close pairs; phase-end orders liquidate all positions.


In [ ]:
coverage = pipeline.prepare_current_inputs(ctx)
display(coverage)
display(ctx["daily"].groupby(["split", "action"]).size().rename("days").reset_index())


## 3. Recompute prediction baselines and verify upstream model outputs

Recompute every mature-label monthly historical mean from the current panel, cross-check both model exports and notebook 02's monthly eligibility, then recompute model/zero/mean forecast metrics on the same scoring samples.


In [ ]:
prediction_comparison = pipeline.recompute_baselines(ctx)
display(prediction_comparison)


## 4. Execute engine and fault checks now

Run signal timing/missing-input/future-information checks, strict entry/exit and reversal boundaries, net-share planning, accounting fixtures, benchmark formulas and threshold-selection logic. These are actual test executions, not loaded pass flags.


In [ ]:
engine_checks = pipeline.run_engine_checks(ctx)
display(engine_checks)


## 5. Recompute validation threshold selection

Replay all six strategies at each of the five registered candidates. Every strategy starts independently from capital 1 and ends flat. Rank the equal-weight mean daily net-minus-cash returns; exact ties prefer the larger entry threshold. The winner is computed, not hardcoded.


In [ ]:
candidate_scores = pipeline.select_threshold(ctx)
display(candidate_scores)
print("THIS run selected entry / exit:", ctx["entry"], "/", ctx["entry"] / 2, "bp")


## 6. Evaluate validation and test, with independent ledger reconstruction

Reuse only the selected validation ledger just computed in this run. Replay all six test strategies from empty positions and capital 1. Independently reconstruct signed dividends, actual net fees, cash carry, borrowing, cash and NAV from source quotes and shares; verify complete intervals, resets and final liquidation.


In [ ]:
pair_metrics = pipeline.run_pair_phases(ctx)
display(pair_metrics[["split", "model", "scheme", "entry_bps", "annualized_net_return",
                      "annualized_net_volatility", "max_drawdown", "annualized_cash_excess_sharpe"]])


## 7. Recompute investable benchmarks

Cash compounds the same available interval rates. Fixed-share SPY reserves the entry fee inside capital 1, credits dividends to cash, earns later cash interest and pays liquidation fees. Each phase starts independently. The reinvestment SPY benchmark for the extension is computed in notebook 06.


In [ ]:
benchmark_metrics = pipeline.run_benchmarks(ctx)
display(benchmark_metrics[["split", "series_id", "annualized_net_return", "max_drawdown", "ending_nav"]])


## 8. Fresh full-phase cost sensitivity

Main plus seven prespecified one-factor changes: execution 0/1/5 bp, annual borrow 0%/2%/3%, or full-cash rebate. Each alternative recomputes NAV-dependent shares and financing under the SAME selected threshold; no scenario is selected by performance. All 96 strategy/phase/scenario metric rows are retained.


In [ ]:
if RUN_COST_SENSITIVITIES:
    sensitivity_metrics = pipeline.run_sensitivities(ctx)
    display(sensitivity_metrics[["split", "scenario_id", "model", "scheme", "annualized_net_return", "max_drawdown"]])
else:
    print("Cost sensitivity deliberately skipped; no old sensitivity file is loaded.")


## 9. Generate new figures and complete the run

Recheck that all input CSVs and engine modules remained unchanged while this run executed. Only after successful completion is this run made available to notebook 06. Notebook autosaves are not financial inputs and do not invalidate the run.


In [ ]:
completion = pipeline.finish_pair_run(ctx)
print("Complete:", RUN_DIR)
print("Fresh replay:", completion["pair_phase_replay"])
print("Historical backtest files used:", completion["historical_backtest_files_used"])
for phase in pipeline.PHASE_OPENS:
    display(Image(filename=str(RUN_DIR / f"figures/pair_{phase}.png")))


## Research appendix: separate training-period OOF diagnostic

The original notebook also displayed a distinct 2012-calibration/2013–2017 model-refitting study. That research extension is not used to generate official forecasts or portfolio returns here. Its only former plumbing dependency—the fixed threshold candidate grid—is registered directly before this run's validation replay. This notebook neither presents nor loads historical OOF results. All main portfolio calculations, baselines, checks and the default full cost sensitivities above are fresh.

For the SPY-core extension and final 12-candidate comparison, continue to **06**. The local `run_config.json` records the complete executable portfolio protocol without an external document dependency.
